In [ ]:
import os
import sys
import subprocess
import time

def log(msg): print(f"\n>>> [MASTER] {msg}", flush=True)

# ------------------------------------------------------------------------------
# 1. ENVIRONMENT SETUP
# ------------------------------------------------------------------------------
log("Initializing Runtime Environment...")

# Uninstall Conflicts
try:
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "tensorflow", "huggingface_hub", "tokenizers"])
except: pass

# Install Dependencies
pkgs = [
    "huggingface_hub>=0.27.0",
    "tokenizers>=0.22.0",
    "safetensors==0.4.5",
    "datasets>=3.1.0",
    "filelock",
    "accelerate>=0.26.0" 
]

try:
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + pkgs + ["--no-deps"])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", 
        "git+https://github.com/huggingface/transformers@main", 
        "--force-reinstall", "--no-deps"
    ])
    log("Dependencies Installed.")
except Exception as e:
    log(f"Setup Failed: {e}")
    sys.exit(1)

# ------------------------------------------------------------------------------
# 2. GENERATE LIBRARY FILES
# ------------------------------------------------------------------------------
log("Generating 'vlm_lib.py'...")

with open("vlm_lib.py", "w") as f:
    f.write(r'''
import torch
import torch.nn as nn
from transformers import PreTrainedModel, PretrainedConfig, Qwen2TokenizerFast, CLIPImageProcessor

# --- CONFIGURATION ---
class VLMConfig(PretrainedConfig):
    model_type = "enterprise_vlm"
    def __init__(self, llm_id="Qwen/Qwen2.5-3B-Instruct", vision_id="openai/clip-vit-large-patch14", image_token_count=256, **kwargs):
        super().__init__(**kwargs)
        self.llm_id = llm_id
        self.vision_id = vision_id
        self.image_token_count = image_token_count

# --- MODEL ARCHITECTURE ---
class Projector(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.Linear(out_dim, out_dim))
    def forward(self, x): return self.net(x)

class EnterpriseVLM(PreTrainedModel):
    config_class = VLMConfig
    _supports_gradient_checkpointing = True 

    def __init__(self, config):
        import transformers
        super().__init__(config)
        self.vision_model = transformers.CLIPVisionModel.from_pretrained(config.vision_id)
        
        llm_config = transformers.AutoConfig.from_pretrained(config.llm_id)
        
        self.llm = transformers.Qwen2ForCausalLM.from_pretrained(
            config.llm_id, 
            config=llm_config, 
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True 
        )
        self.projector = Projector(self.vision_model.config.hidden_size, self.llm.config.hidden_size)
        self._freeze()

    def _freeze(self):
        self.vision_model.requires_grad_(False)
        self.llm.requires_grad_(False)
        self.projector.requires_grad_(True)
        self.projector.to(dtype=torch.bfloat16)

    def get_input_embeddings(self):
        return self.llm.get_input_embeddings()

    # [CRITICAL FIX] Manually forward the enable command to the inner model
    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None):
        self.llm.gradient_checkpointing_enable(gradient_checkpointing_kwargs=gradient_checkpointing_kwargs)

    def forward(self, input_ids, labels, pixel_values, **kwargs):
        use_cache = False if self.training else True
        
        with torch.no_grad():
            # CHANGE: Enable output_hidden_states to true in the forward call
            outputs = self.vision_model(pixel_values, output_hidden_states=True)
            # CHANGE: Access penultimate layer (-2). 
            # Note: outputs.hidden_states is a tuple. -1 is final layer, -2 is penultimate.
            # We skip the CLS token ([:, 1:]) just like before.
            vis = outputs.hidden_states[-2][:, 1:]
        
        vis = self.projector(vis.to(dtype=torch.bfloat16))

        if vis.shape[1] != self.config.image_token_count:
            vis = torch.nn.functional.adaptive_avg_pool1d(
                vis.transpose(1, 2), self.config.image_token_count
            ).transpose(1, 2)

        txt_emb = self.llm.get_input_embeddings()(input_ids)
        combined_emb = torch.cat((vis, txt_emb[:, self.config.image_token_count:]), dim=1)
        
        return self.llm(inputs_embeds=combined_emb, labels=labels, use_cache=use_cache)

# --- ROBUST DATA PROCESSING ---
class Processor:
    def __init__(self, tokenizer, img_proc, token_count, max_len=2048):
        self.tok = tokenizer
        self.img = img_proc
        self.tokens = token_count
        self.max_len = max_len 
        self.pad_id = tokenizer.pad_token_id

    def __call__(self, item):
        if "messages" not in item: return None
        try:
            if "images" in item and item["images"] and len(item["images"]) > 0:
                image = item["images"][0].convert("RGB")
                px = self.img(image, return_tensors="pt").pixel_values[0]
            else:
                return None

            full_text = ""
            for msg in item["messages"]:
                role = msg["role"]
                content = msg["content"]
                
                if role == "user": full_text += "<|im_start|>user\n"
                elif role == "assistant": full_text += "<|im_start|>assistant\n"
                
                if isinstance(content, str):
                    full_text += content
                elif isinstance(content, list):
                    for part in content:
                        if part["type"] == "text":
                            full_text += part["text"]
                
                full_text += "<|im_end|>\n"
            
            full_text += "<|endoftext|>"

            text_cap = self.max_len - self.tokens
            ids = self.tok(
                full_text, 
                max_length=text_cap, 
                truncation=True, 
                padding="max_length", 
                return_tensors="pt",
                add_special_tokens=False
            ).input_ids[0]

            full_ids = torch.cat([torch.tensor([self.pad_id]*self.tokens, dtype=torch.long), ids])
            
            labels = full_ids.clone()
            labels[:self.tokens] = -100 
            labels[labels == self.pad_id] = -100 

            return {"input_ids": full_ids, "labels": labels, "pixel_values": px}
        except Exception as e:
            print(f"Processor Error: {e}", flush=True)
            return None
''')

log("Generating 'run_task.py'...")
with open("run_task.py", "w") as f:
    f.write(r'''import os
import sys

# [CRITICAL] This function runs on EVERY TPU CORE.
# The patch MUST be applied here to work.
def _mp_fn(index):
    import os
    import torch
    import torch_xla
    
    # [FIX] Apply Monkey-Patch inside the worker process!
    # This tricks PyTorch into finding 'torch.xla' for checkpointing.
    if not hasattr(torch, "xla"):
        torch.xla = torch_xla

    import torch_xla.core.xla_model as xm
    from datasets import load_dataset
    from transformers import TrainingArguments, Trainer, TrainerCallback
    from filelock import FileLock
    import vlm_lib 
    
    device = xm.xla_device()
    if index == 0: print(f"[CORE {index}] TPU Initialized: {device}", flush=True)

    torch.manual_seed(99)
    
    conf = vlm_lib.VLMConfig()
    tok = vlm_lib.Qwen2TokenizerFast.from_pretrained(conf.llm_id)
    
    if tok.pad_token is None:
        tok.pad_token_id = 151643 
        
    img_proc = vlm_lib.CLIPImageProcessor.from_pretrained(conf.vision_id)
    
    if index == 0: print(">>> Loading Dataset...", flush=True)
    with FileLock("/tmp/data_load.lock"):
        raw_ds = load_dataset("HuggingFaceH4/llava-instruct-mix-vsft", split="train")
    
    proc = vlm_lib.Processor(tok, img_proc, conf.image_token_count, max_len=2048)
    
    class CleanDataset(torch.utils.data.Dataset):
        def __init__(self, ds):
            self.ds = ds
        def __len__(self): return len(self.ds)
        def __getitem__(self, idx):
            res = proc(self.ds[idx])
            if res is not None: return res
            return proc(self.ds[0])

    train_ds = CleanDataset(raw_ds)
    if index == 0:
        print("\n>>> DATASET SANITY CHECK (Sample 0) <<<", flush=True)
        try:
            sample = train_ds[0]
            ids = sample["input_ids"]
            # Filter out padding (-100 or pad_token) for clean reading
            valid_ids = ids[ids != tok.pad_token_id] 
            decoded_text = tok.decode(valid_ids)
            
            print(f"Token Length: {len(ids)}")
            print(f"Decoded Text:\n{decoded_text}\n", flush=True)
            
            if len(valid_ids) < 10:
                print(">>> WARNING: Sample looks suspiciously empty!", flush=True)
        except Exception as e:
            print(f">>> CRITICAL DATASET ERROR: {e}", flush=True)
    # ------------------------------
    model = vlm_lib.EnterpriseVLM(conf).to(device)
    ckpt_path = "/kaggle/input/clipqwenstep10000/projector_v2.pt" 
    
    if os.path.exists(ckpt_path):
        if index == 0: print(f">>> RESUMING FROM: {ckpt_path}", flush=True)
        # Load weights onto CPU first, then model moves them to TPU automatically
        state = torch.load(ckpt_path, map_location="cpu")
        model.projector.load_state_dict(state)
    else:
        if index == 0: print(f">>> WARNING: {ckpt_path} NOT FOUND. Starting fresh.", flush=True)
    class PrinterCallback(TrainerCallback):
        def on_log(self, args, state, control, logs=None, **kwargs):
            if state.is_local_process_zero:
                output = {k: v for k, v in logs.items() if k in ['loss', 'epoch', 'learning_rate']}
                if output:
                    print(f"[Step {state.global_step}] Stats: {output}", flush=True)



    
    args = TrainingArguments(
        output_dir="/kaggle/working/checkpoints",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        max_steps=40000,
        learning_rate=1e-4,
        max_grad_norm=0.5,
        num_train_epochs=1,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        save_strategy="no",
        save_total_limit=1,
        logging_steps=5,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="adafactor",
        disable_tqdm=True ,
        dataloader_pin_memory=False,
        dataloader_num_workers=0, 
        remove_unused_columns=False,
        report_to="none",
        local_rank=index,
        ddp_find_unused_parameters=False,
    )
    
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, callbacks=[PrinterCallback])

    if index == 0: print(">>> Starting Training Loop...", flush=True)
    trainer.train()
    
    if index == 0:
        print(">>> Saving Adapter...", flush=True)
        state = {k: v.cpu() for k, v in model.projector.state_dict().items()}
        torch.save(state, "projector_v3.pt")

if __name__ == "__main__":
    os.environ["PJRT_DEVICE"] = "TPU"
    os.environ["XLA_USE_BF16"] = "1"
    os.environ["WANDB_DISABLED"] = "true"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    
    print(">>> [MASTER] Ensuring model files are cached...", flush=True)
    try:
        from huggingface_hub import snapshot_download
        snapshot_download("Qwen/Qwen2.5-3B-Instruct")
        snapshot_download("openai/clip-vit-large-patch14")
    except Exception as e:
        print(f"Pre-download warning: {e}")

    import torch_xla.distributed.xla_multiprocessing as xmp
    print(">>> [LAUNCHER] Spawning XLA...", flush=True)
    xmp.spawn(_mp_fn, args=(), start_method='spawn')
''')

log("Setup Complete. Ready to Train.")

In [ ]:
import subprocess
import sys
import os
import time

def execute_safe():
    print(">>> [MASTER] Launching Training Process...", flush=True)
    
    # 1. Kill old zombies
    # This ensures no previous 'run_task.py' processes are hogging the TPU
    try: 
        subprocess.call("pkill -9 -f run_task.py", shell=True)
        time.sleep(2) # Give the OS a moment to reclaim resources
    except: pass
    
    # 2. Prepare clean environment
    # Crucial: We remove TPU env vars from THIS parent process 
    # to avoid accidental initialization in the notebook kernel.
    clean_env = os.environ.copy()
    keys_to_purge = ["PJRT_DEVICE", "XLA_USE_BF16", "TPU_PROCESS_ADDRESSES"]
    for k in keys_to_purge:
        if k in clean_env: del clean_env[k]

    # 3. Launch
    # We use sys.executable to ensure we use the same Python interpreter
    try:
        subprocess.check_call([sys.executable, "run_task.py"], env=clean_env)
        print(">>> [MASTER] TRAINING COMPLETE SUCCESS.", flush=True)
    except subprocess.CalledProcessError as e:
        print(f"!!! [MASTER] Training Crashed with code {e.returncode}", flush=True)

# Run the execution
if __name__ == "__main__":
    execute_safe()